# Concert counts from setlist.fm, 2010–2020

Download concert counts from setlist.fm: setlists logged per city per year.

Before running:
  1. Make a free account at setlist.fm.
  2. Apply for an API key: setlist.fm → log in → Settings → API (apps)
     (https://www.setlist.fm/settings/apps). It's free for non-commercial use.
  3. Save the key in a file called setlist_key.txt next to this notebook (see Settings).

Outputs (saved next to this notebook):
  concert_cities.csv          the setlist.fm city matched to each city you listed
  concert_counts.csv          setlists per city per year
  concert_world_totals.csv    setlists worldwide per year

It only asks for counts (one request per city per year), not every setlist, and it saves
as it goes. If it stops (rate limit, lost connection), just run it again: anything already
saved is skipped.

  pip install requests pandas

In [2]:
import os
import time

import pandas as pd
import requests

## Settings
Paste your API key here. Edit the years or the city list if you like.

In [4]:
# The key is read from a separate file so it never ends up in the notebook (or on GitHub).
# Create a text file called setlist_key.txt next to this notebook containing only your key,
# and add setlist_key.txt to your .gitignore.
KEY_FILE = "setlist_key.txt"
API_KEY = (open(KEY_FILE).read().strip() if os.path.exists(KEY_FILE)
           else os.environ.get("SETLISTFM_API_KEY", "your_api_key_here"))
YEARS = range(2010, 2021)        # 2010–2020; use range(2010, 2024) to include the recovery
PAUSE_S = 1.0                    # wait between requests, to stay within the rate limit

CITIES_CSV = "concert_cities.csv"
COUNTS_CSV = "concert_counts.csv"
WORLD_CSV = "concert_world_totals.csv"

# (city name, 2-letter country code, optional state/region to tell same-named places apart).
# Edit freely. If a city still matches the wrong place, add its setlist.fm city ID to
# MANUAL_IDS (the ID is the number at the end of the city's page URL on setlist.fm).
CITIES = [
    # North America
    ("New York", "US", "New York"), ("Los Angeles", "US", "California"),
    ("Chicago", "US", "Illinois"), ("Nashville", "US", "Tennessee"),
    ("Austin", "US", "Texas"), ("Las Vegas", "US", "Nevada"),
    ("Toronto", "CA", "Ontario"), ("Mexico City", "MX", ""),
    # South America
    ("São Paulo", "BR", "São Paulo"), ("Buenos Aires", "AR", ""), ("Santiago", "CL", ""),
    ("Bogotá", "CO", ""),
    # Europe
    ("London", "GB", "England"), ("Manchester", "GB", "England"), ("Glasgow", "GB", "Scotland"),
    ("Dublin", "IE", ""), ("Paris", "FR", "Île-de-France"), ("Berlin", "DE", ""),
    ("Hamburg", "DE", ""), ("Amsterdam", "NL", ""), ("Brussels", "BE", ""),
    ("Madrid", "ES", ""), ("Barcelona", "ES", ""), ("Milan", "IT", ""),
    ("Stockholm", "SE", ""), ("Oslo", "NO", ""), ("Copenhagen", "DK", ""),
    ("Helsinki", "FI", ""), ("Warsaw", "PL", ""), ("Prague", "CZ", ""),
    ("Vienna", "AT", ""), ("Moscow", "RU", ""),
    # Africa and the Middle East
    ("Johannesburg", "ZA", ""), ("Lagos", "NG", ""), ("Cairo", "EG", ""),
    ("Dubai", "AE", ""), ("Tel Aviv", "IL", ""), ("Istanbul", "TR", ""),
    # Asia and Oceania
    ("Tokyo", "JP", ""), ("Osaka", "JP", ""), ("Seoul", "KR", ""), ("Beijing", "CN", ""),
    ("Shanghai", "CN", ""), ("Hong Kong", "HK", ""), ("Taipei", "TW", ""),
    ("Mumbai", "IN", ""), ("Singapore", "SG", ""), ("Bangkok", "TH", ""),
    ("Kuala Lumpur", "MY", ""), ("Jakarta", "ID", ""), ("Manila", "PH", ""),
    ("Ho Chi Minh City", "VN", ""),
    ("Sydney", "AU", "New South Wales"), ("Melbourne", "AU", "Victoria"),
    ("Auckland", "NZ", ""),
]

# Force a specific setlist.fm city ID, e.g. {("Paris", "FR"): "2988507"}
MANUAL_IDS = {}

MAX_PAGES = 3          # pages of city search results to look through (20 per page)
CHECK_YEAR = 2019      # when several same-named cities fit, keep the one with most setlists here

BASE = "https://api.setlist.fm/rest/1.0"

## API helpers

In [6]:
session = requests.Session()
session.headers.update({"x-api-key": API_KEY, "Accept": "application/json"})


def get(path, params, tries=5):
    """GET with polite pauses. Returns parsed JSON, or None when setlist.fm finds nothing."""
    for attempt in range(tries):
        time.sleep(PAUSE_S)
        r = session.get(f"{BASE}/{path}", params=params, timeout=30)
        if r.status_code == 200:
            return r.json()
        if r.status_code == 404:          # setlist.fm answers "not found" for zero results
            return None
        if r.status_code == 429:          # too many requests: wait longer and retry
            wait = 10 * (attempt + 1)
            print(f"  rate limited, waiting {wait} s ...")
            time.sleep(wait)
            continue
        if r.status_code in (401, 403):
            raise SystemExit("setlist.fm refused the key. Check API_KEY.")
        print(f"  HTTP {r.status_code} for {path} {params}; retrying")
        time.sleep(5)
    raise RuntimeError(f"Gave up on {path} {params}")


def load(path, columns):
    return pd.read_csv(path, dtype={"city_id": str}) if os.path.exists(path) else pd.DataFrame(columns=columns)

## 1. Match each city to its setlist.fm ID
Each line printed shows the place it picked, so check the states and regions look right. If one is wrong, put its setlist.fm city ID in `MANUAL_IDS`, delete that row from `concert_cities.csv`, and run again.

In [8]:
import unicodedata


def plain(text):
    """Lower case without accents, so 'Bogotá' matches 'Bogota'."""
    text = unicodedata.normalize("NFKD", str(text or ""))
    return "".join(ch for ch in text if not unicodedata.combining(ch)).lower().strip()


def city_candidates(name, cc, state):
    found = []
    for page in range(1, MAX_PAGES + 1):
        data = get("search/cities", {"name": name, "country": cc, "p": page})
        if not data:
            break
        found += data.get("cities", [])
        if page * data.get("itemsPerPage", 20) >= data.get("total", 0):
            break
    same_name = [c for c in found if plain(c["name"]) == plain(name)]
    if state:
        same_state = [c for c in same_name if plain(state) in plain(c.get("state", ""))]
        same_name = same_state or same_name
    return same_name


def setlists_in(city_id, year):
    data = get("search/setlists", {"cityId": city_id, "year": year})
    return data.get("total", 0) if data else 0


def match_cities():
    cols = ["query", "country", "city_id", "name", "state", "lat", "lon"]
    done = load(CITIES_CSV, cols)
    have = set(zip(done["query"], done["country"]))
    rows = done.to_dict("records")
    for name, cc, state in CITIES:
        if (name, cc) in have:
            continue
        cands = city_candidates(name, cc, state)
        if (name, cc) in MANUAL_IDS:
            data = get(f"city/{MANUAL_IDS[(name, cc)]}", {})
            cands = [data] if data else []
        if not cands:
            print(f"No setlist.fm city called {name} found in {cc}; add it to MANUAL_IDS")
            continue
        if len(cands) > 1:
            # several places share the name: keep the one with the most concerts
            counts = [(setlists_in(c["id"], CHECK_YEAR), c) for c in cands[:4]]
            pick = max(counts, key=lambda t: t[0])[1]
        else:
            pick = cands[0]
        rows.append({"query": name, "country": cc, "city_id": pick["id"],
                     "name": pick["name"], "state": pick.get("state", ""),
                     "lat": pick["coords"]["lat"], "lon": pick["coords"]["long"]})
        print(f"Matched {name}, {cc} -> {pick['name']} ({pick.get('state', '')}) id {pick['id']}"
              + (f"  [picked from {len(cands)} same-named places]" if len(cands) > 1 else ""))
        pd.DataFrame(rows).to_csv(CITIES_CSV, index=False)
    return pd.DataFrame(rows)

## 2. Count setlists per city per year

In [10]:
def count_concerts(cities):
    done = load(COUNTS_CSV, ["city_id", "name", "country", "year", "setlists"])
    have = set(zip(done["city_id"].astype(str), done["year"].astype(int)))
    rows = done.to_dict("records")
    todo = [(c, y) for c in cities.itertuples() for y in YEARS
            if (str(c.city_id), y) not in have]
    print(f"{len(todo)} city-year counts to fetch")
    for i, (c, y) in enumerate(todo, 1):
        data = get("search/setlists", {"cityId": c.city_id, "year": y})
        total = data.get("total", 0) if data else 0
        rows.append({"city_id": c.city_id, "name": c.name, "country": c.country,
                     "year": y, "setlists": total})
        print(f"[{i}/{len(todo)}] {c.name} {y}: {total}")
        pd.DataFrame(rows).to_csv(COUNTS_CSV, index=False)
    return pd.DataFrame(rows)

## 3. World totals per year

In [12]:
def world_totals():
    done = load(WORLD_CSV, ["year", "setlists"])
    have = set(done["year"].astype(int))
    rows = done.to_dict("records")
    for y in YEARS:
        if y in have:
            continue
        data = get("search/setlists", {"year": y})
        total = data.get("total", 0) if data else 0
        rows.append({"year": y, "setlists": total})
        print(f"World {y}: {total}")
        pd.DataFrame(rows).to_csv(WORLD_CSV, index=False)
    return pd.DataFrame(rows)


def main():
    if API_KEY.startswith("PASTE"):
        raise SystemExit("No API key found. Put it in setlist_key.txt next to this notebook.")
    cities = match_cities()
    counts = count_concerts(cities)
    world = world_totals()
    print("\nDone. Setlists per year across your cities:")
    print(counts.groupby("year")["setlists"].sum().to_string())
    print("\nWorld totals:")
    print(world.sort_values("year").to_string(index=False))

## Run
Safe to re-run: anything already saved is skipped.

In [14]:
main()

Matched New York, US -> New York (New York) id 5128581
Matched Los Angeles, US -> Los Angeles (California) id 5368361
Matched Chicago, US -> Chicago (Illinois) id 4887398
Matched Nashville, US -> Nashville (Tennessee) id 4644585
Matched Austin, US -> Austin (Texas) id 4671654
Matched Las Vegas, US -> Las Vegas (Nevada) id 5506956
Matched Toronto, CA -> Toronto (Ontario) id 6167865
Matched Mexico City, MX -> Mexico City (Distrito Federal) id 3530597
Matched São Paulo, BR -> São Paulo (São Paulo) id 3448439
Matched Buenos Aires, AR -> Buenos Aires (Autonomous City of Buenos Aires) id 3435910
Matched Santiago, CL -> Santiago (Región Metropolitana) id 3871336
Matched Bogotá, CO -> Bogota (Bogota D.C.) id 3688689
Matched London, GB -> London (England) id 2643743
Matched Manchester, GB -> Manchester (England) id 2643123
Matched Glasgow, GB -> Glasgow (Scotland) id 2648579
Matched Dublin, IE -> Dublin (Dublin) id 2964574
Matched Paris, FR -> Paris (Île-de-France) id 2988507  [picked from 2 sa